In [ ]:
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
import glob
import json
import random

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils import data
from tqdm import tqdm
import yaml

from nets import nn
from utils import util
from utils.valid import (
    compute_point_label_metrics_single,
    visualize_ground_truth_and_prediction_separately_detail_single,
    plot_training_progress,
    plot_hierarchy_progress,
)
from utils.hierarchical import (
    PDL1HierarchicalComputeLoss,
    load_flat_checkpoint_into_pdl1_hierarchical,
)

from utils.ihc_dataset import (
    IHCHierarchicalDataset,
    PDL1_CLASS_COLORS,
    PDL1_CLASS_NAMES,
    discover_records,
    slide_id_from_record,
    split_records_by_slide_balanced,
    count_record_classes,
)

GPU_ID = 1  # PD-L1 GPU
if torch.cuda.is_available():
    if GPU_ID >= torch.cuda.device_count():
        raise RuntimeError(
            f"GPU_ID={GPU_ID}를 요청했지만 사용 가능한 GPU는 {torch.cuda.device_count()}개입니다."
        )
    torch.cuda.set_device(GPU_ID)
    device = torch.device(f"cuda:{GPU_ID}")
    print(f"device: {device} | {torch.cuda.get_device_name(GPU_ID)}")
else:
    device = torch.device("cpu")
    print("device: cpu")
with open('utils/detail_args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)

In [ ]:
input_size = 512

# ── MediautoAI PD-L1 stomach 데이터 ─────────────────────────────
data_root = '../../data/mediautoAI_PDL1_Stomach/'
resize_size = 1024

# JSON class_name이 아래 7개에 해당할 때만 사용합니다.
class_names = PDL1_CLASS_NAMES.copy()
num_classes = len(class_names)
params['names'] = class_names

records = discover_records(data_root)
image_filenames = [str(image_path) for image_path, _ in records]

print(f"📂 {len(records)} image/label pairs found")
print(f"   image: {image_filenames[0]}")
print(f"   label: {records[0][1]}")

# 지원하지 않는 class_name(예: Tumor)은 여기서 제외됩니다.
first_labels = IHCHierarchicalDataset.load_labels(records[0][1])
print(f"✅ first label shape: {first_labels.shape}")

In [ ]:
# 기존 collate_fn1 이름을 유지합니다.
def collate_fn1(batch):
    return IHCHierarchicalDataset.collate_fn(batch)


# ── Train / Val split ────────────────────────────────────────────
# Slide leakage를 막으면서 모든 class가 충분히 포함되도록 전체 slide 조합을
# 비교해 validation class coverage가 가장 균형적인 3장을 선택합니다.
VALIDATION_SLIDE_COUNT = 3
train_records, val_records = split_records_by_slide_balanced(
    records, val_slide_count=VALIDATION_SLIDE_COUNT
)
train_slides = {slide_id_from_record(record) for record in train_records}
val_slides = {slide_id_from_record(record) for record in val_records}
assert train_slides.isdisjoint(val_slides)
print(f"Selected validation slides ({len(val_slides)}):")
for slide_name in sorted(val_slides):
    print(f"  - {slide_name}")

# 80%의 crop은 현재 잘 학습되지 않는 class 1~5 중 하나를 먼저 고른 뒤,
# 해당 세포가 crop 중앙부에 들어오도록 위치를 정합니다. 풍부한 class 0/6은
# crop 안에 자연스럽게 충분히 포함되므로 target class 선택에서는 제외합니다.
CLASS_AWARE_CROP_PROBABILITY = 0.8
CROP_CLASS_SAMPLING_WEIGHTS = [1, 1, 1, 1, 1, 1, 1]

train_dataset = IHCHierarchicalDataset(
    train_records, input_size=input_size, resize_size=resize_size,
    augment=True, random_crop=True, cache_images=True,
    class_aware_crop_probability=CLASS_AWARE_CROP_PROBABILITY,
    class_sampling_weights=CROP_CLASS_SAMPLING_WEIGHTS
)
val_dataset = IHCHierarchicalDataset(
    val_records, input_size=input_size, resize_size=resize_size,
    augment=False, random_crop=False, cache_images=True, four_crop=True
)

# ── 클래스 분포 분석 (기존 노트북과 동일한 위치) ────────────────
print("\n" + "=" * 70)
print("📊 클래스 분포 분석")
print("=" * 70)

total_counts = count_record_classes(train_records).astype(np.float64)
val_counts = count_record_classes(val_records).astype(np.float64)

for i in range(num_classes):
    coverage = val_counts[i] / (total_counts[i] + val_counts[i]) * 100
    print(f"  [{i}] {class_names[i]:20s}: "
          f"train={int(total_counts[i]):7d} | val={int(val_counts[i]):6d} | "
          f"val coverage={coverage:5.1f}%")

print("\nPD-L1 hierarchical label 해석:")
print(f"  전체 7 classes   : {int(total_counts.sum()):,}")
print(f"  other            : {int(total_counts[6]):,}")
print("  ※ WeightedRandomSampler는 사용하지 않고 class-aware crop을 적용합니다.")
print("  ※ Root: Epithelial / Immune / other")
print("  ※ Epithelial: Negative / Positive")
print("  ※ Immune: Lymphocyte / Macrophage × Negative / Positive")

# Branch별 sqrt imbalance weight. 세부 loss는 해당 root GT에만 적용됩니다.
epithelial_count = total_counts[0] + total_counts[3]
immune_count = total_counts[1] + total_counts[2] + total_counts[4] + total_counts[5]
other_count = total_counts[6]
root_counts = np.array([epithelial_count, immune_count, other_count])
root_weights = np.sqrt(root_counts.max() / np.maximum(root_counts, 1))
epithelial_positive_weight = np.sqrt(total_counts[0] / max(total_counts[3], 1))
immune_lymphocyte_weight = np.sqrt(
    (total_counts[2] + total_counts[5]) / max(total_counts[1] + total_counts[4], 1)
)
immune_negative_weight = np.sqrt(
    (total_counts[4] + total_counts[5]) / max(total_counts[1] + total_counts[2], 1)
)
hierarchy_params = {
    **params,
    'objectness': 1.0, 'root': 2.0, 'epithelial': 1.0,
    'immune_type': 2.0, 'immune_polarity': 2.0,
    'root_class_weights': root_weights.tolist(),
    'epithelial_positive_weight': float(epithelial_positive_weight),
    'immune_lymphocyte_weight': float(immune_lymphocyte_weight),
    'immune_negative_weight': float(immune_negative_weight),
    'objectness_focal_gamma': 1.5,
}
VALIDATION_CONFIDENCE_THRESHOLD = 0.10
print(f"  Class-aware crop : {CLASS_AWARE_CROP_PROBABILITY:.0%}")
print(f"  Root counts/weights: {root_counts.astype(int).tolist()} / {root_weights.round(3).tolist()}")
print(f"  Epithelial positive weight: {epithelial_positive_weight:.3f}")
print(f"  Immune lymphocyte weight  : {immune_lymphocyte_weight:.3f}")
print(f"  Immune negative weight    : {immune_negative_weight:.3f}")
print(f"  Validation confidence     : {VALIDATION_CONFIDENCE_THRESHOLD:.2f}")
print("=" * 70)

# ── DataLoader ───────────────────────────────────────────────────
# 작은 데이터셋이므로 한 epoch에서 전체 train set을 50번 순회합니다.
# 같은 dataset 객체를 참조하므로 1024 image cache는 중복되지 않고,
# 각 접근마다 새로운 random crop/augmentation이 적용됩니다.
EPOCH_DATA_REPEATS = 50
train_epoch_dataset = torch.utils.data.ConcatDataset(
    [train_dataset] * EPOCH_DATA_REPEATS
)
batch_size = 16
train_loader = torch.utils.data.DataLoader(
    train_epoch_dataset, batch_size=batch_size, shuffle=True,
    num_workers=4, collate_fn=collate_fn1, drop_last=False,
    pin_memory=(device.type == 'cuda'), persistent_workers=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, collate_fn=collate_fn1, drop_last=False,
    pin_memory=(device.type == 'cuda'), persistent_workers=True,
)

print(f"\n📦 DataLoaders ready:")
print(f"  Train patches : {len(train_dataset):,} ({len(train_slides)} slides)")
print(f"  Passes/epoch  : {EPOCH_DATA_REPEATS}회")
print(f"  Samples/epoch : {len(train_epoch_dataset):,}")
print(f"  Val 1024 patches: {len(val_records):,} ({len(val_slides)} slides)")
print(f"  Val 512 tiles  : {len(val_dataset):,} ({len(val_records)} × 4)")
print(f"  Train batches : {len(train_loader):,}")
print(f"  Val batches   : {len(val_loader):,}")
cache_mib = (train_dataset.cache_memory_bytes + val_dataset.cache_memory_bytes) / 1024**2
print(f"  1024 image cache: {cache_mib:.1f} MiB")

In [ ]:
def visualize_sample_with_overlay(dataset, index=0):
    image_tensor, cls_tensor, box_tensor, _ = dataset[index]

    image = image_tensor.numpy().transpose(1, 2, 0)
    cls = cls_tensor.numpy()
    boxes = box_tensor.numpy()
    height, width = image.shape[:2]

    colors = [PDL1_CLASS_COLORS[i] for i in range(num_classes)]

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(image.astype(np.uint8))

    for i in range(len(boxes)):
        cx_norm, cy_norm, w_norm, h_norm = boxes[i]
        cx_px = cx_norm * width
        cy_px = cy_norm * height
        w_px = w_norm * width
        h_px = h_norm * height
        x1 = cx_px - w_px / 2
        y1 = cy_px - h_px / 2

        class_id = int(cls[i])
        rect = patches.Rectangle(
            (x1, y1), w_px, h_px, linewidth=2,
            edgecolor=colors[class_id], facecolor='none'
        )
        ax.add_patch(rect)

    ax.set_title(f'Sample {index} | Total Objects: {len(boxes)}', fontsize=12)
    ax.axis('off')

    legend_elements = [
        patches.Patch(
            color=colors[i],
            label=f'{class_names[i]} ({int((cls == i).sum())})'
        ) for i in range(num_classes)
    ]
    fig.legend(
        handles=legend_elements, loc='lower center', ncol=3,
        bbox_to_anchor=(0.5, -0.05), fontsize=9
    )
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()

    print(f"\n📊 Sample {index}:")
    print(f"  이미지 크기 : {width} x {height}")
    print(f"  총 객체 수  : {len(boxes)}")
    for i in range(num_classes):
        count = int((cls == i).sum())
        if count > 0:
            print(f"  [{i}] {class_names[i]:20s}: {count}개")


visualize_sample_with_overlay(train_dataset, index=0)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Epithelial / Immune / other 계층형 YOLO 구성 ─────────────────
LEARNING_RATE = 5e-4
MIN_LEARNING_RATE = 1e-6
MAX_EPOCHS = 2000

model = nn.yolo_v11_m_pdl1_hierarchical().to(device)
print(f"PID={os.getpid()} | declared={device} | model={next(model.parameters()).device}")

# 기존 balanced flat 모델에서 backbone/FPN/box/semantic feature와
# 각 branch의 초기 decision boundary를 전이합니다. optimizer는 새로 시작합니다.
# WARM_START_CHECKPOINT = (
#     '../../model/mediautoAI_PDL1_Stomach/'
#     'pdl1_flat_7class_balanced_yolov11/best_model.pt'
# )
# if WARM_START_CHECKPOINT and os.path.exists(WARM_START_CHECKPOINT):
#     transfer_report = load_flat_checkpoint_into_pdl1_hierarchical(
#         model, WARM_START_CHECKPOINT, map_location=device
#     )
#     print(f"✅ Flat warm start: {transfer_report}")
# else:
#     print("ℹ️ Warm-start checkpoint 없음: scratch에서 시작합니다.")

optimizer = torch.optim.AdamW(
    util.set_params(model, params['weight_decay']),
    lr=LEARNING_RATE, betas=(0.9, 0.999),
    weight_decay=params['weight_decay']
)
criterion = PDL1HierarchicalComputeLoss(model, hierarchy_params)

cosine_scheduler = CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE
)

# 기존 노트북과 동일한 one-batch 확인
model.train()
smoke_images, smoke_targets = next(iter(train_loader))
smoke_images = smoke_images.to(device).float() / 255.0
with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
    smoke_outputs = model(smoke_images)
    smoke_losses = criterion(smoke_outputs, smoke_targets)
print("\n✅ PD-L1 hierarchical loss smoke test:")
print("  " + " | ".join(
    f"{name}={value.item():.4f}" for name, value in smoke_losses.items()
))
del smoke_images, smoke_targets, smoke_outputs
del smoke_losses
if device.type == 'cuda':
    torch.cuda.empty_cache()

print(f"\n🎯 PD-L1 hierarchical 학습 설정:")
print("  ✅ Root: Epithelial / Immune / other")
print("  ✅ Branch: Epi polarity / Immune type / Immune polarity")
print(f"  ✅ Learning rate   : {LEARNING_RATE} → {MIN_LEARNING_RATE}")
print(f"  ✅ Epochs          : {MAX_EPOCHS} (early stopping disabled)")
print(f"  ✅ Data passes/epoch: {EPOCH_DATA_REPEATS}")
print(f"  ✅ Classes         : {num_classes}")

In [ ]:
train_losses = []
train_box_losses = []
train_root_losses = []
train_epithelial_losses = []
train_immune_type_losses = []
train_immune_polarity_losses = []
train_dfl_losses = []
val_det_recalls = []
val_cls_accs = []
val_macro_f1s = []
val_macro_precisions = []
val_macro_recalls = []
val_class_stats_history = []
val_hierarchy_history = []

epochs = MAX_EPOCHS
save_dir = '../../model/mediautoAI_PDL1_Stomach/pdl1_hierarchical/'
os.makedirs(save_dir, exist_ok=True)

start_epoch = 0
best_validation_score = 0.0
accumulate = max(round(64 / batch_size), 1)
amp_scale = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

# ── 체크포인트 불러오기 (재개 시 주석 해제) ─────────────────────
# checkpoint_path = os.path.join(save_dir, 'last_model.pt')
# if os.path.exists(checkpoint_path):
#     checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
#     if checkpoint.get('architecture') != 'pdl1_hierarchical_eio':
#         raise ValueError('Epithelial/Immune/other 계층형 checkpoint가 아닙니다.')
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     cosine_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
#     start_epoch = checkpoint['epoch'] + 1
#     amp_scale.load_state_dict(checkpoint.get('amp_scale_state_dict', {}))
#     best_validation_score = checkpoint.get('best_validation_score', 0.0)

print(f"Gradient accumulation steps: {accumulate}")

for epoch in range(start_epoch, epochs):
    model.train()

    avg_box_loss = util.AverageMeter()
    avg_objectness_loss = util.AverageMeter()
    avg_root_loss = util.AverageMeter()
    avg_epithelial_loss = util.AverageMeter()
    avg_immune_type_loss = util.AverageMeter()
    avg_immune_polarity_loss = util.AverageMeter()
    avg_dfl_loss = util.AverageMeter()
    avg_total_loss = util.AverageMeter()

    train_pbar = tqdm(
        enumerate(train_loader), total=len(train_loader),
        desc=f'Epoch {epoch + 1}/{epochs} Training'
    )
    optimizer.zero_grad(set_to_none=True)

    for i, (torch_images, targets) in train_pbar:
        torch_images = torch_images.to(device, non_blocking=True).float() / 255.0

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(torch_images)
            losses = criterion(outputs, targets)
            unscaled_total_loss = losses['total']
            total_loss = unscaled_total_loss / accumulate

        avg_box_loss.update(losses['box'].item(), torch_images.size(0))
        avg_objectness_loss.update(losses['objectness'].item(), torch_images.size(0))
        avg_root_loss.update(losses['root'].item(), torch_images.size(0))
        avg_epithelial_loss.update(losses['epithelial'].item(), torch_images.size(0))
        avg_immune_type_loss.update(losses['immune_type'].item(), torch_images.size(0))
        avg_immune_polarity_loss.update(losses['immune_polarity'].item(), torch_images.size(0))
        avg_dfl_loss.update(losses['dfl'].item(), torch_images.size(0))
        avg_total_loss.update(unscaled_total_loss.item(), torch_images.size(0))

        amp_scale.scale(total_loss).backward()
        if (i + 1) % accumulate == 0 or (i + 1) == len(train_loader):
            amp_scale.step(optimizer)
            amp_scale.update()
            optimizer.zero_grad(set_to_none=True)

        memory = (
            f'{torch.cuda.memory_reserved() / 1E9:.4g}G'
            if device.type == 'cuda' else 'CPU'
        )
        status = (
            f'Memory: {memory} | Box: {avg_box_loss.avg:.3f} | '
            f'Obj: {avg_objectness_loss.avg:.3f} | Root: {avg_root_loss.avg:.3f} | '
            f'Epi: {avg_epithelial_loss.avg:.3f} | '
            f'IType: {avg_immune_type_loss.avg:.3f} | '
            f'IPol: {avg_immune_polarity_loss.avg:.3f} | DFL: {avg_dfl_loss.avg:.3f}'
        )
        train_pbar.set_description(f'Epoch {epoch + 1}/{epochs} | {status}')

    cosine_scheduler.step()

    train_losses.append(avg_total_loss.avg)
    train_box_losses.append(avg_box_loss.avg)
    train_root_losses.append(avg_root_loss.avg)
    train_epithelial_losses.append(avg_epithelial_loss.avg)
    train_immune_type_losses.append(avg_immune_type_loss.avg)
    train_immune_polarity_losses.append(avg_immune_polarity_loss.avg)
    train_dfl_losses.append(avg_dfl_loss.avg)

    # ── Validation: final 7-class + hierarchy branch metrics ───────
    try:
        point_metrics = compute_point_label_metrics_single(
            model, val_loader, device, params, distance_threshold=16,
            class_agnostic_nms=True,
            confidence_threshold=VALIDATION_CONFIDENCE_THRESHOLD
        )
        detection_recall = point_metrics.get('detection_recall', 0)
        cls_accuracy = point_metrics.get('classification_accuracy', 0)
        macro_precision = point_metrics.get('macro_precision', 0)
        macro_recall = point_metrics.get('macro_recall', 0)
        macro_f1 = point_metrics.get('macro_f1', 0)
        overall_recall = point_metrics.get('overall_recall', 0)
        class_stats = point_metrics.get('class_stats', {})
        root_epithelial_recall = point_metrics.get('root_epithelial_recall', 0)
        root_immune_recall = point_metrics.get('root_immune_recall', 0)
        root_other_recall = point_metrics.get('root_other_recall', 0)
        root_balanced_accuracy = point_metrics.get('root_balanced_accuracy', 0)
        epithelial_branch_accuracy = point_metrics.get('epithelial_branch_accuracy', 0)
        immune_type_accuracy = point_metrics.get('immune_type_accuracy', 0)
        immune_polarity_accuracy = point_metrics.get('immune_polarity_accuracy', 0)

        val_det_recalls.append(detection_recall)
        val_cls_accs.append(cls_accuracy)
        val_macro_f1s.append(macro_f1)
        val_macro_precisions.append(macro_precision)
        val_macro_recalls.append(macro_recall)
        val_class_stats_history.append(class_stats)
        val_hierarchy_history.append({
            'root_epithelial_recall': root_epithelial_recall,
            'root_immune_recall': root_immune_recall,
            'root_other_recall': root_other_recall,
            'root_balanced_accuracy': root_balanced_accuracy,
            'epithelial_branch_accuracy': epithelial_branch_accuracy,
            'immune_type_accuracy': immune_type_accuracy,
            'immune_polarity_accuracy': immune_polarity_accuracy,
        })

        # 최종 7-class 성능을 우선하고 root/detection을 보조로 사용합니다.
        validation_score = (
            0.65 * macro_f1 + 0.20 * detection_recall
            + 0.15 * root_balanced_accuracy
        )

        print(f"\nEpoch {epoch + 1}/{epochs}:")
        print(
            f"  Train Loss — Box: {avg_box_loss.avg:.4f} | "
            f"Obj: {avg_objectness_loss.avg:.4f} | Root: {avg_root_loss.avg:.4f} | "
            f"Epi: {avg_epithelial_loss.avg:.4f} | "
            f"IType: {avg_immune_type_loss.avg:.4f} | "
            f"IPol: {avg_immune_polarity_loss.avg:.4f} | DFL: {avg_dfl_loss.avg:.4f}"
        )
        print(f"  Detection Recall       : {detection_recall:.4f}")
        print(f"  Classification Accuracy: {cls_accuracy:.4f}")
        print(f"  Macro Precision        : {macro_precision:.4f}")
        print(f"  Macro Recall           : {macro_recall:.4f}")
        print(f"  Macro F1               : {macro_f1:.4f}")
        print(f"  Root Recall Epi/Imm/Other: {root_epithelial_recall:.4f} / {root_immune_recall:.4f} / {root_other_recall:.4f}")
        print(f"  Root Balanced Accuracy : {root_balanced_accuracy:.4f}")
        print(f"  Epithelial Polarity Acc: {epithelial_branch_accuracy:.4f}")
        print(f"  Immune Type Accuracy   : {immune_type_accuracy:.4f}")
        print(f"  Immune Polarity Acc    : {immune_polarity_accuracy:.4f}")
        print(f"  Validation Score       : {validation_score:.4f}")
    except Exception as e:
        print(f"Validation 오류: {e}")
        detection_recall = cls_accuracy = 0
        macro_precision = macro_recall = macro_f1 = overall_recall = 0
        validation_score = 0
        root_epithelial_recall = root_immune_recall = root_other_recall = 0
        root_balanced_accuracy = epithelial_branch_accuracy = 0
        immune_type_accuracy = immune_polarity_accuracy = 0
        class_stats = {}

    # ── Checkpoint ──────────────────────────────────────────────
    improved = validation_score > best_validation_score
    if improved:
        best_validation_score = validation_score

    ckpt_data = {
        'architecture': 'pdl1_hierarchical_eio',
        'hierarchy': 'root(Epithelial,Immune,other); Epi polarity; Immune type+polarity',
        'num_classes': num_classes,
        'epoch_data_repeats': EPOCH_DATA_REPEATS,
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': cosine_scheduler.state_dict(),
        'amp_scale_state_dict': amp_scale.state_dict(),
        'train_loss': avg_total_loss.avg,
        'box_loss': avg_box_loss.avg,
        'objectness_loss': avg_objectness_loss.avg,
        'root_loss': avg_root_loss.avg,
        'epithelial_loss': avg_epithelial_loss.avg,
        'immune_type_loss': avg_immune_type_loss.avg,
        'immune_polarity_loss': avg_immune_polarity_loss.avg,
        'dfl_loss': avg_dfl_loss.avg,
        'detection_recall': detection_recall,
        'classification_accuracy': cls_accuracy,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'overall_recall': overall_recall,
        'root_epithelial_recall': root_epithelial_recall,
        'root_immune_recall': root_immune_recall,
        'root_other_recall': root_other_recall,
        'root_balanced_accuracy': root_balanced_accuracy,
        'epithelial_branch_accuracy': epithelial_branch_accuracy,
        'immune_type_accuracy': immune_type_accuracy,
        'immune_polarity_accuracy': immune_polarity_accuracy,
        'validation_score': validation_score,
        'best_validation_score': best_validation_score,
        'class_stats': class_stats,
        'params': hierarchy_params,
    }
    torch.save(ckpt_data, os.path.join(save_dir, 'last_model.pt'))

    if improved:
        torch.save(ckpt_data, os.path.join(save_dir, 'best_model.pt'))
        print(f"🎉 새로운 베스트! Validation Score: {validation_score:.4f}")

    # ── 시각화 (10 epoch마다) ────────────────────────────────────
    if (epoch + 1) % 10 == 0:
        try:
            sample_idx = random.randint(0, len(val_dataset) - 1)
            visualize_ground_truth_and_prediction_separately_detail_single(
                model, val_dataset, idx=sample_idx,
                epoch=epoch + 1, save_dir=save_dir,
                conf_threshold=VALIDATION_CONFIDENCE_THRESHOLD
            )
        except Exception as e:
            print(f"시각화 오류: {e}")

    # ── 학습 곡선 (10 epoch마다) ────────────────────────────────
    if (epoch + 1) % 10 == 0:
        try:
            plot_training_progress(
                train_losses, val_det_recalls, val_cls_accs,
                val_macro_precisions, val_macro_recalls, val_macro_f1s,
                epoch + 1, save_dir,
                class_stats_history=val_class_stats_history
            )
            plot_hierarchy_progress(val_hierarchy_history, epoch + 1, save_dir)
        except Exception as e:
            print(f"그래프 오류: {e}")

print("🎯 PD-L1 Epithelial/Immune/other 계층형 학습 완료!")
print(f"  Best Validation Score : {best_validation_score:.4f}")
print(f"  저장 경로              : {save_dir}")